In [0]:
%pip install pytest

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pytest

from pyspark.sql.functions import *

In [0]:
CATALOG = "credit_risk_analysis_db"
SCHEMA = "bronze_sch"

TABLE = f"{CATALOG}.{SCHEMA}.bronze_applicant_profiles"

In [0]:
def test_table_exists():

    assert spark.catalog.tableExists(TABLE), \
        f"{TABLE} does not exist"

test_table_exists()

print(" Test Passed - Bronze table exists")

✅ Test Passed - Bronze table exists


In [0]:
BASE_PATH = "abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/"

SOURCE_PATH = BASE_PATH + "applicant_profiles.parquet"

source_df = spark.read.parquet(SOURCE_PATH)

bronze_df = spark.table(TABLE)

assert source_df.count() == bronze_df.count(), \
    "Row Count Validation Failed"

print(" Row Count Validation Passed")

✅ Row Count Validation Passed


In [0]:
# ===========================
# Test 3 : Duplicate Check
# ===========================

bronze_df = spark.table(TABLE)

duplicate_count = (
    bronze_df
    .groupBy("applicant_id")
    .count()
    .filter("count > 1")
    .count()
)

assert duplicate_count == 0, \
    f" Found {duplicate_count} duplicate applicant_id values."

print(" Duplicate Check Passed")

✅ Duplicate Check Passed


In [0]:
# ===========================
# Test 4 : Null Validation
# ===========================

bronze_df = spark.table(TABLE)

null_count = bronze_df.filter(
    col("applicant_id").isNull()
).count()

assert null_count == 0, \
    f" Found {null_count} NULL applicant_id values."

print("Null Validation Passed")

Null Validation Passed


In [0]:
# ===========================
# Test 5 : Metadata Validation
# ===========================

bronze_df = spark.table(TABLE)

metadata_nulls = bronze_df.filter(
    col("ingestion_timestamp").isNull() |
    col("source_file").isNull() |
    col("batch_id").isNull() |
    col("load_date").isNull()
).count()

assert metadata_nulls == 0, \
    f" Found {metadata_nulls} rows with missing metadata."

print(" Metadata Validation Passed")

✅ Metadata Validation Passed


In [0]:
# ===========================
# Test 6 : Schema Validation
# ===========================

expected_columns = [
    "applicant_id",
    "gender",
    "age",
    "income",
    "business_or_commercial",
    "occupancy_type",
    "construction_type",
    "total_units",
    "region",
    "ingestion_timestamp",
    "source_file",
    "batch_id",
    "load_date"
]

actual_columns = spark.table(TABLE).columns

assert actual_columns == expected_columns, \
    f" Schema mismatch.\nExpected: {expected_columns}\nActual: {actual_columns}"

print("Schema Validation Passed")

✅ Schema Validation Passed


In [0]:
BRONZE_TABLES = {

    "bronze_applicant_profiles": {
        "source_file": "applicant_profiles.parquet",
        "primary_key": "applicant_id",
        "check_duplicates": True
    },

    "bronze_credit_application": {
        "source_file": "credit_application.parquet",
        "primary_key": "applicant_id",
        "check_duplicates": False
    },

    "bronze_credit_history": {
        "source_file": "credit_history.parquet",
        "primary_key": "applicant_id",
        "check_duplicates": False
    },

    "bronze_loan_details": {
        "source_file": "loan_details.parquet",
        "primary_key": "loan_id",
        "check_duplicates": True
    },

    "bronze_economic_indicators": {
        "source_file": "economic_indicators.parquet",
        "primary_key": ["year","region"],
        "check_duplicates": False
    }

}

In [0]:
BASE_PATH = "abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/"

CATALOG = "credit_risk_analysis_db"
SCHEMA = "bronze_sch"

In [0]:
from pyspark.sql.functions import *

def validate_bronze_table(
        table_name,
        source_file,
        primary_key,
        check_duplicates):

    print("="*70)
    print(f"Testing {table_name}")
    print("="*70)

    table = f"{CATALOG}.{SCHEMA}.{table_name}"

    source_path = BASE_PATH + source_file

    source_df = spark.read.parquet(source_path)

    bronze_df = spark.table(table)

    # -------------------------------------------------
    # Test 1 : Table Exists
    # -------------------------------------------------

    assert spark.catalog.tableExists(table), \
        f"{table} does not exist"

    print(" Table Exists")

    # -------------------------------------------------
    # Test 2 : Row Count
    # -------------------------------------------------

    source_count = source_df.count()
    bronze_count = bronze_df.count()

    assert source_count == bronze_count, \
        f"Row Count Mismatch Source={source_count} Bronze={bronze_count}"

    print(" Row Count")

    # -------------------------------------------------
    # Test 3 : Duplicate Check
    # -------------------------------------------------

    if check_duplicates:

        if isinstance(primary_key, list):

            duplicate_count = (
                bronze_df
                .groupBy(*primary_key)
                .count()
                .filter("count > 1")
                .count()
            )

        else:

            duplicate_count = (
                bronze_df
                .groupBy(primary_key)
                .count()
                .filter("count > 1")
                .count()
            )

        assert duplicate_count == 0, \
            f"Duplicate Records Found = {duplicate_count}"

        print(" Duplicate Check")

    else:

        print("⏭ Duplicate Check Skipped")

    # -------------------------------------------------
    # Test 4 : Null Check
    # -------------------------------------------------

    if isinstance(primary_key, list):

        condition = None

        for c in primary_key:

            if condition is None:
                condition = col(c).isNull()
            else:
                condition = condition | col(c).isNull()

        null_count = bronze_df.filter(condition).count()

    else:

        null_count = bronze_df.filter(
            col(primary_key).isNull()
        ).count()

    assert null_count == 0, \
        f"NULL values found in Primary Key = {null_count}"

    print(" Null Check")

    # -------------------------------------------------
    # Test 5 : Metadata Check
    # -------------------------------------------------

    metadata_count = bronze_df.filter(

        col("ingestion_timestamp").isNull() |
        col("source_file").isNull() |
        col("batch_id").isNull() |
        col("load_date").isNull()

    ).count()

    assert metadata_count == 0, \
        f"Metadata NULL Count = {metadata_count}"

    print(" Metadata Check")

    print(f" {table_name} Passed Successfully\n")

In [0]:
for table_name, config in BRONZE_TABLES.items():

    validate_bronze_table(

        table_name=table_name,

        source_file=config["source_file"],

        primary_key=config["primary_key"],

        check_duplicates=config["check_duplicates"]

    )

Testing bronze_applicant_profiles
 Table Exists
 Row Count
 Duplicate Check
 Null Check
 Metadata Check
 bronze_applicant_profiles Passed Successfully

Testing bronze_credit_application
 Table Exists
 Row Count
⏭ Duplicate Check Skipped
 Null Check
 Metadata Check
 bronze_credit_application Passed Successfully

Testing bronze_credit_history
 Table Exists
 Row Count
⏭ Duplicate Check Skipped
 Null Check
 Metadata Check
 bronze_credit_history Passed Successfully

Testing bronze_loan_details
 Table Exists
 Row Count
 Duplicate Check
 Null Check
 Metadata Check
 bronze_loan_details Passed Successfully

Testing bronze_economic_indicators
 Table Exists
 Row Count
⏭ Duplicate Check Skipped
 Null Check
 Metadata Check
 bronze_economic_indicators Passed Successfully



In [0]:
tables = [
    "bronze_applicant_profiles",
    "bronze_credit_application",
    "bronze_credit_history",
    "bronze_loan_details",
    "bronze_economic_indicators"
]

for table in tables:
    print("\n" + "="*80)
    print(table)
    print("="*80)

    df = spark.table(f"credit_risk_analysis_db.bronze_sch.{table}")

    print("Columns:")
    print(df.columns)

    display(df.limit(5))


bronze_applicant_profiles
Columns:
['applicant_id', 'gender', 'age', 'income', 'business_or_commercial', 'occupancy_type', 'construction_type', 'total_units', 'region', 'ingestion_timestamp', 'source_file', 'batch_id', 'load_date']


applicant_id,gender,age,income,business_or_commercial,occupancy_type,construction_type,total_units,region,ingestion_timestamp,source_file,batch_id,load_date
24890,Sex Not Available,25-34,1740.0,nob/c,pr,sb,1U,south,2026-07-09T18:11:19.706Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,abb9f09e-1ed7-4394-b366-0db17709beb6,2026-07-09
24891,Male,55-64,4980.0,b/c,pr,sb,1U,North,2026-07-09T18:11:19.706Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,abb9f09e-1ed7-4394-b366-0db17709beb6,2026-07-09
24892,Male,35-44,9480.0,nob/c,pr,sb,1U,south,2026-07-09T18:11:19.706Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,abb9f09e-1ed7-4394-b366-0db17709beb6,2026-07-09
24893,Male,45-54,11880.0,nob/c,pr,sb,1U,North,2026-07-09T18:11:19.706Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,abb9f09e-1ed7-4394-b366-0db17709beb6,2026-07-09
24894,Joint,25-34,10440.0,nob/c,pr,sb,1U,North,2026-07-09T18:11:19.706Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,abb9f09e-1ed7-4394-b366-0db17709beb6,2026-07-09



bronze_credit_application
Columns:
['applicant_id', 'year', 'loan_limit', 'approv_in_adv', 'loan_type', 'loan_purpose', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'submission_of_application', 'region', 'application_status', 'extra_col1', 'extra_col2', 'extra_col3', 'extra_col4', 'ingestion_timestamp', 'source_file', 'batch_id', 'load_date']


applicant_id,year,loan_limit,approv_in_adv,loan_type,loan_purpose,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,submission_of_application,region,application_status,extra_col1,extra_col2,extra_col3,extra_col4,ingestion_timestamp,source_file,batch_id,load_date
24890,2019,cf,nopre,type1,p1,116500,null,null,null,360,to_inst,south,1,null,null,null,null,2026-07-09T18:11:25.697Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,565424c2-1699-4b4b-ac89-f2f6f72cccb9,2026-07-09
24891,2019,cf,nopre,type2,p1,206500,null,null,null,360,to_inst,North,1,null,null,null,null,2026-07-09T18:11:25.697Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,565424c2-1699-4b4b-ac89-f2f6f72cccb9,2026-07-09
24892,2019,cf,pre,type1,p1,406500,4.56,0.2,595,360,to_inst,south,0,null,null,null,s,2026-07-09T18:11:25.697Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,565424c2-1699-4b4b-ac89-f2f6f72cccb9,2026-07-09
24893,2019,cf,nopre,type1,p4,456500,4.25,0.681,null,360,not_inst,North,0,null,null,null,null,2026-07-09T18:11:25.697Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,565424c2-1699-4b4b-ac89-f2f6f72cccb9,2026-07-09
24894,2019,cf,pre,type1,p1,696500,4,0.3042,0,360,not_inst,North,0,null,null,null,null,2026-07-09T18:11:25.697Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,565424c2-1699-4b4b-ac89-f2f6f72cccb9,2026-07-09



bronze_credit_history
Columns:
['applicant_id', 'credit_worthiness', 'open_credit', 'credit_type', 'credit_score', 'co_applicant_credit_type', 'negative_amortization', 'interest_only', 'lump_sum_payment', 'debt_to_income_ratio', 'ingestion_timestamp', 'source_file', 'batch_id', 'load_date']


applicant_id,credit_worthiness,open_credit,credit_type,credit_score,co_applicant_credit_type,negative_amortization,interest_only,lump_sum_payment,debt_to_income_ratio,ingestion_timestamp,source_file,batch_id,load_date
24890,l1,nopc,EXP,758,CIB,not_neg,not_int,not_lpsm,45.0,2026-07-09T18:11:29.309Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,92e2190d-ad8e-49c6-924f-62f020e7ee5f,2026-07-09
24891,l1,nopc,EQUI,552,EXP,not_neg,not_int,lpsm,null,2026-07-09T18:11:29.309Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,92e2190d-ad8e-49c6-924f-62f020e7ee5f,2026-07-09
24892,l1,nopc,EXP,834,CIB,neg_amm,not_int,not_lpsm,46.0,2026-07-09T18:11:29.309Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,92e2190d-ad8e-49c6-924f-62f020e7ee5f,2026-07-09
24893,l1,nopc,EXP,587,CIB,not_neg,not_int,not_lpsm,42.0,2026-07-09T18:11:29.309Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,92e2190d-ad8e-49c6-924f-62f020e7ee5f,2026-07-09
24894,l1,nopc,CRIF,602,EXP,not_neg,not_int,not_lpsm,39.0,2026-07-09T18:11:29.309Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,92e2190d-ad8e-49c6-924f-62f020e7ee5f,2026-07-09



bronze_loan_details
Columns:
['applicant_id', 'loan_id', 'loan_type', 'loan_purpose', 'loan_amount', 'loan_term', 'property_value', 'loan_to_value', 'secured_by', 'security_type', 'ingestion_timestamp', 'source_file', 'batch_id', 'load_date']


applicant_id,loan_id,loan_type,loan_purpose,loan_amount,loan_term,property_value,loan_to_value,secured_by,security_type,ingestion_timestamp,source_file,batch_id,load_date
24890,24890,type1,p1,116500,360.0,118000.0,98.72881356,home,direct,2026-07-09T18:11:32.402Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,790ead45-a1ee-4d8d-8a38-12c59a8f9df6,2026-07-09
24891,24891,type2,p1,206500,360.0,null,null,home,direct,2026-07-09T18:11:32.402Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,790ead45-a1ee-4d8d-8a38-12c59a8f9df6,2026-07-09
24892,24892,type1,p1,406500,360.0,508000.0,80.01968504,home,direct,2026-07-09T18:11:32.402Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,790ead45-a1ee-4d8d-8a38-12c59a8f9df6,2026-07-09
24893,24893,type1,p4,456500,360.0,658000.0,69.3768997,home,direct,2026-07-09T18:11:32.402Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,790ead45-a1ee-4d8d-8a38-12c59a8f9df6,2026-07-09
24894,24894,type1,p1,696500,360.0,758000.0,91.88654354,home,direct,2026-07-09T18:11:32.402Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,790ead45-a1ee-4d8d-8a38-12c59a8f9df6,2026-07-09



bronze_economic_indicators
Columns:
['year', 'region', 'avg_property_value', 'avg_interest_rate', 'Interest_rate_spread', 'loan_status', 'ingestion_timestamp', 'source_file', 'batch_id', 'load_date']


year,region,avg_property_value,avg_interest_rate,Interest_rate_spread,loan_status,ingestion_timestamp,source_file,batch_id,load_date
2019,south,118000.0,null,null,1,2026-07-09T18:11:35.224Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,68080040-6572-44e2-8a3f-88826ca61f85,2026-07-09
2019,North,null,null,null,1,2026-07-09T18:11:35.224Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,68080040-6572-44e2-8a3f-88826ca61f85,2026-07-09
2019,south,508000.0,4.56,0.2,0,2026-07-09T18:11:35.224Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,68080040-6572-44e2-8a3f-88826ca61f85,2026-07-09
2019,North,658000.0,4.25,0.681,0,2026-07-09T18:11:35.224Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,68080040-6572-44e2-8a3f-88826ca61f85,2026-07-09
2019,North,758000.0,4.0,0.3042,0,2026-07-09T18:11:35.224Z,abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,68080040-6572-44e2-8a3f-88826ca61f85,2026-07-09
